# 🌲 Modelo Random Forest Balanceado — Predicción de IAAS
## Versión con Exportación del Modelo, Gráficos y Set de Prueba
## v5 — Criterios validados con lista oficial IAAS

**Objetivo:** Entrenar el modelo, evaluar su desempeño con gráficos completos, exportar el modelo `.pkl` y exportar el 20% de datos de prueba a Excel.

---

### ¿Qué genera este notebook?
1. **`modelo_iaas.pkl`** — modelo listo para usar en el frontend
2. **`matriz_confusion.png`** — matriz de confusión (conteos y porcentajes)
3. **`grafico_confianza.png`** — curva ROC + distribución de probabilidades
4. **`importancia_variables.png`** — importancia de cada variable
5. **`set_prueba_con_predicciones.xlsx`** — el 20% de test con columnas originales y predicciones

---
## 📦 PASO 1: Instalación de librerías

In [ ]:
!pip install imbalanced-learn openpyxl --quiet
print("✅ Librerías instaladas")

---
## 📂 PASO 2: Subir los archivos Excel a Colab

In [ ]:
from google.colab import files

print("Sube los dos archivos Excel (Data_1_25_v2.xlsx y Data_2_24_v2.xlsx)")
uploaded = files.upload()
print("\nArchivos subidos:", list(uploaded.keys()))

---
## 📥 PASO 3: Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    f1_score, precision_score, recall_score
)
from imblearn.over_sampling import SMOTE

print("✅ Librerías importadas correctamente")

---
## 🔍 PASO 4: Extracción de datos desde los archivos Excel

Los datos están organizados en hojas mensuales. Buscamos la fila que contiene `'Rut'` para identificar el encabezado real en cada hoja.

In [ ]:
MESES_2025 = [
    'ENERO 2025', 'FEBRERO 2025', 'MARZO 2025', 'ABRIL 2025',
    'MAYO 2025', 'JUNIO 2025', 'JULIO 2025', 'AGOSTO 2025',
    'SEPTIEMBRE 2025', 'OCTUBRE 2025', 'NOVIEMBRE 2025', 'DICIEMBRE 2025'
]

MESES_2024 = [
    'ENERO 24', 'FEBRERO 24', 'MARZO 24', 'ABRIL 24',
    'MAYO 24', 'JUNIO 24', 'JULIO 2024', 'AGOSTO 2024',
    'SEPTIEMBRE 2024', 'OCTUBRE 2024', 'NOVIEMBRE 2024', 'DICIEMBRE 2024'
]

COLUMNAS_MODELO = [
    'Servicio', 'Procedencia', 'Destino', 'Tipo de invasivo',
    'ARAISP', 'Criterio 1', 'Criterio 2', 'Criterio 3',
    'Criterio 4', 'Criterio 5', 'IAAS'
]

# Todas las columnas del Excel de entrada, en el orden exacto original
COLUMNAS_TODAS = [
    'Rut', 'Rut_formato', 'Servicio', 'Apellido paterno', 'Apellido materno',
    'Nombres', 'Fecha de ingreso', 'Fecha de egreso', 'Procedencia', 'Destino',
    'Tipo de invasivo', 'Fecha de instalación', 'Fecha de retiro',
    'ARAISP', 'Criterio 1', 'Criterio 2', 'Criterio 3', 'Criterio 4', 'Criterio 5', 'IAAS'
]

# Diccionario global: índice de fila en df → fila completa del Excel original
filas_originales = {}


def extraer_hoja(archivo, nombre_hoja, anio):
    """
    Extrae los datos de una hoja mensual del Excel.
    Busca la fila que contiene 'Rut' para identificar el encabezado real.
    Guarda las filas completas (todas las columnas) en filas_originales.
    """
    try:
        df_raw = pd.read_excel(archivo, sheet_name=nombre_hoja, header=None)
    except Exception:
        return None

    fila_encabezado = None
    for i, fila in df_raw.iterrows():
        if any(str(v).strip() == 'Rut' for v in fila.values):
            fila_encabezado = i
            break

    if fila_encabezado is None:
        print(f"  ⚠️  No se encontró encabezado en hoja: {nombre_hoja}")
        return None

    nombres_col = [str(v).strip() for v in df_raw.iloc[fila_encabezado].values]
    datos = df_raw.iloc[fila_encabezado + 1:].copy()
    datos.columns = range(len(nombres_col))
    idx_col = {nombre: i for i, nombre in enumerate(nombres_col)}

    # Extraer columnas del modelo
    resultado = {}
    for col in COLUMNAS_MODELO:
        if col in idx_col:
            resultado[col] = datos[idx_col[col]].values
        else:
            resultado[col] = np.nan

    resultado['anio'] = anio
    resultado['mes_hoja'] = nombre_hoja

    # Guardar las filas completas (todas las columnas originales) indexadas
    # por la posición que tendrán en el df global una vez concatenado
    df_parcial = pd.DataFrame(resultado)
    df_parcial['_archivo'] = archivo

    # Las 20 columnas originales en orden: dos columnas 'Rut' se llaman Rut y Rut_formato
    nombres_orig = list(nombres_col)  # nombres reales del encabezado
    for fila_local_idx, fila_datos in datos.iterrows():
        fila_dict = {}
        for j, nombre in enumerate(nombres_orig):
            fila_dict[nombre] = fila_datos[j]
        df_parcial._iloc_origen = fila_local_idx  # no usado directamente
        # guardamos indexado por posición local en esta hoja
        if '_filas_raw' not in extraer_hoja.__dict__:
            extraer_hoja._filas_raw = []
        extraer_hoja._filas_raw.append(fila_dict)

    return df_parcial


print("📂 Cargando datos de 2025...")
dfs = []
extraer_hoja._filas_raw = []  # reset
for hoja in MESES_2025:
    d = extraer_hoja('Data_1_25_v2.xlsx', hoja, 2025)
    if d is not None:
        dfs.append(d)
        print(f"  ✅ {hoja}: {len(d)} registros")

print("\n📂 Cargando datos de 2024...")
for hoja in MESES_2024:
    d = extraer_hoja('Data_2_24_v2.xlsx', hoja, 2024)
    if d is not None:
        dfs.append(d)
        print(f"  ✅ {hoja}: {len(d)} registros")

df = pd.concat(dfs, ignore_index=True)

# Construir DataFrame con todas las columnas originales, alineado con df
df_raw_completo = pd.DataFrame(extraer_hoja._filas_raw).reset_index(drop=True)
print(f"\n✅ Total de registros cargados: {len(df)}")
print(f"   Columnas originales preservadas: {list(df_raw_completo.columns)}")

---
## 🧹 PASO 5: Limpieza y preprocesamiento

- Limpieza de la variable objetivo (IAAS)
- Normalización de texto y unificación de variantes
- Creación de variables derivadas (`n_criterios`, `tiene_araisp`, `tipo_dispositivo_cat`)
- Codificación de categóricas con LabelEncoder

In [ ]:
# --- 5.1 Limpieza de variable objetivo ---
def limpiar_iaas(valor):
    if pd.isna(valor):
        return np.nan
    v = str(valor).strip().lower()
    if v in ('si', 'sí'):
        return 1
    elif v == 'no':
        return 0
    return np.nan

df['IAAS_binaria'] = df['IAAS'].apply(limpiar_iaas)
df = df.dropna(subset=['IAAS_binaria']).copy()
df['IAAS_binaria'] = df['IAAS_binaria'].astype(int)

print("📊 Distribución IAAS limpia (0=No, 1=Sí):")
conteo = df['IAAS_binaria'].value_counts()
print(conteo)
print(f"\nBalance: {conteo[1]/(conteo[0]+conteo[1])*100:.1f}% casos positivos (IAAS=Sí)")

In [ ]:
# --- 5.2 Ingeniería de características ---
cols_texto = ['Servicio', 'Procedencia', 'Destino', 'Tipo de invasivo']
for col in cols_texto:
    df[col] = df[col].astype(str).str.strip().str.lower()
    df[col] = df[col].replace({'nan': 'sin_dato', '': 'sin_dato'})

df['Tipo de invasivo'] = df['Tipo de invasivo'].replace({
    'epicutáneo': 'catéter epicutáneo',
    'dve': 'derivativa ventricular externa',
    'dvp': 'derivativa ventricular peritoneal'
})

# ============================================================
# LISTA OFICIAL DE CRITERIOS IAAS
# Extraída de la hoja 'Criterios' de Data_1_25_v2.xlsx y Data_2_24_v2.xlsx
# ============================================================
CRITERIOS_VALIDOS = [
    # A - Bacteriemia/Fungemia asociada a CVC
    'A.I.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'A.I.a.e2.Hipotermia igual o menor a 36 °C axilar',
    'A.I.a.e3.Hipotensión',
    'A.I.a.e4. Taquicardia o bradicardia',
    'A.I.a.e5.Apnea en pacientes menores de un año',
    'A.I.a.e6.Eritema y exudado en sitio de inserción del CVC',
    'A.I.b.e1.Detección en uno o más set de hemocultivos periféricos de un microorganismo patógeno no relacionado con otra infección activa en otra localización por el mismo agente',
    'A.I.b.e2.Detección de microorganismo comensal en al menos dos sets de hemocultivos periféricos tomados en sitios anatómicos diferentes no relacionado con otra infección activa en otra localización por el mismo agente',
    'A.I.b.e3.Detección de microorganismo comensal en al menos un set de hemocultivos periféricos y en cultivo de punta de catéter retirado por sospecha clínica de infección, no relacionado con otra infección activa en otra localización por el mismo agente',
    # B - ITU asociada a sonda vesical
    'B.I.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'B.I.a.e2.Síntomas irritativos vesicales (tenesmo vesical, urgencia miccional, polaquiuria, disuria, dolor suprapúbico)',
    'B.I.a.e3.Dolor costo vertebral a la palpación o espontáneo',
    'B.I.a.e4.Alteración nueva del estado de conciencia en pacientes de 65 o más años',
    'B.I.b.e1.Leucocituria de acuerdo con los valores de referencia del laboratorio que procesó la muestra tomada',
    'B.I.b.e2.Presencia de placas de pus',
    'B.I.b.e3.Presencia de piocitos',
    'B.I.c.e1.Cultivo de orina con no más de dos microorganismos, en el que al menos uno de ellos tiene recuento de más de 100.000 UFC/ml',
    # C - Infección Sitio Quirúrgico
    'C.I.a.e1.Presencia de pus (exudado purulento) en el sitio de incisión quirúrgica, incluido el sitio de la salida de drenaje por contrabertura, con o sin cultivos positivos',
    'C.II.a.e1.Fiebre igual o mayor a 38 °C no atribuible a otra causa',
    'C.II.a.e2.Sensibilidad o dolor en la zona de la incisión quirúrgica',
    'C.II.a.e3.Aumento de volumen localizado en la zona de la incisión quirúrgica',
    'C.II.a.e4.Eritema o calor local en la zona de la incisión quirúrgica',
    'C.II.a.e5.La incisión es deliberadamente abierta por un integrante del equipo de salud1 con presencia de exudado que, sin tener aspecto de pus, se describe como turbio, serohemático o seropurulento',
    'C.II.a.e6.Aislamiento de microorganismo en cultivo obtenido con técnica aséptica de la incisión o tejido subcutáneo',
    # D - Diarrea infecciosa
    'D.I.a.e1.Paciente tiene dos o más deposiciones líquidas dentro de 12 horas con o sin otra sintomatología, no atribuible a causas no infecciosas',
    'D.I.b.e1.Si se cuenta con agente etiológico identificado, no hay evidencias que el microorganismo se haya encontrado presente o en periodo incubación al momento del ingreso hospitalario',
    'D.II.a.e1.Paciente presenta un episodio de deposiciones líquidas o disgregadas',
    'D.II.b.e1.Crecimiento de microorganismo patógeno entérico en cultivo de deposiciones o en muestra de hisopado rectal',
    'D.II.b.e2.Microorganismo patógeno entérico detectado por cualquier medio que no sea cultivo',
    'D.II.c.e1.No hay evidencias que el microorganismo se haya encontrado presente o en periodo incubación al momento del ingreso hospitalario',
    # E - C. difficile
    'E.I.a.e1.Presencia de más de una deposición líquida en 12 horas',
    'E.I.a.e2.Presencia de 3 o más deposiciones disgregadas o líquidas en 24 horas',
    'E.II.a.e1.Muestra de deposición positiva a toxina de C. difficile por cualquier técnica de laboratorio, o aislamiento de cepa productora de toxina detectada en deposición por cultivo u otro medio incluida biología molecular',
    # F - Neumonía
    'F.I.a1.e1.Infiltrado',
    'F.I.a1.e2.Condensación',
    'F.I.a1.e3.Cavitación',
    'F.I.a2.e1.Infiltrado nuevo o progresión de uno existente',
    'F.I.a2.e2.Condensación',
    'F.I.a2.e3.Cavitación',
    'F.I.b.e1.Fiebre mayor o igual a 38 °C axilar',
    'F.I.b.e2.Leucopenia (<4.000 leucocitos/mm3) o leucocitosis (>12.000 leucocitos/mm3)',
    'F.I.b.e3.Deterioro en el intercambio gaseoso no explicable por otra causa',
    'F.I.b.e4.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC/ml o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    'F.II.a.e1.Infiltrado nuevo o progresión de uno existente',
    'F.II.a.e2.Condensación',
    'F.II.a.e3.Cavitación',
    'F.II.a.e4.Neumatoceles',
    'F.II.b.e1.Deterioro en el intercambio gaseoso no explicable por otra causa',
    'F.II.c.e1.Temperatura corporal inestable',
    'F.II.c.e2.Leucopenia (11.000 leucocitos/mm3) con desviación a izquierda (Mayor o igual a 10% de baciliformes o formas más inmaduras)',
    'F.II.c.e3.Aparición de expectoración purulenta, o cambios en las características, o aumento de la cantidad, o aumento en los requerimientos de aspiración de secreciones',
    'F.II.c.e4.Sibilancias, estertores o roncus',
    'F.II.c.e5.Inestabilidad hemodinámica',
    'F.II.c.e6.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC/ml1010 o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    'F.III.a.e1.Presenta Deterioro en el intercambio gaseoso no explicable por otra causa',
    'F.III.b.e1.Aparición de expectoración, aumento o cambio en las características, o aumento de los requerimientos de aspiración o succión de secreciones',
    'F.III.b.e2.Hemoptisis',
    'F.III.b.e3.Aspirado endotraqueal con aislamiento de microorganismo patógeno > 100.000 UFC3/ml o lavado bronco alveolar o cepillo protegido con recuento significativo (104 o 103 ufc/ml respectivamente) o panel molecular con recuento significativo para neumonía de acuerdo con laboratorio local',
    # G - IRA viral
    'G.I.a.e1.Fiebre igual o mayor a 38 °C axilar o hipotermia sin otra causa reconocible',
    'G.I.a.e2.Leucopenia (<4.000 leucocitos/mm3) o leucocitosis (>11.000 leucocitos/mm3)',
    'G.I.a.e3.Tos',
    'G.I.a.e4.Aparición o incremento de producción de expectoración',
    'G.I.a.e5.Roncus',
    'G.I.a.e6.Sibilancias',
    'G.I.a.e7.Distrés respiratorio o síndrome de dificultad respiratoria',
    'G.I.a.e8.Apnea',
    'G.I.a.e9.Bradicardia',
    'G.I.a.e10.Imagen pulmonar no presente al ingreso compatible con infección viral',
    'G.I.b.e1.Detección de agente viral respiratorio por cualquier técnica de laboratorio',
    'G.I.c.e1.No hay evidencias que el agente viral respiratorio se haya encontrado presente o en periodo incubación al momento del ingreso hospitalario',
    # H - COVID-19
    'H.I.a.e1.Fiebre igual o mayor a 37,8 °C axilar',
    'H.I.a.e2.Perdida brusca y completa del olfato (anosmia)',
    'H.I.a.e3.Perdida brusca y completa del gusto (ageusia)',
    'H.I.a.e4.Tos o estornudos',
    'H.I.a.e5.Congestión nasal',
    'H.I.a.e6.Disnea o dificultad respiratoria',
    'H.I.a.e7.Taquipnea',
    'H.I.a.e8.Odinofagia',
    'H.I.a.e9.Mialgia',
    'H.I.a.e10.Debilidad general o fatiga',
    'H.I.a.e11.Dolor torácico',
    'H.I.a.e12.Calofríos',
    'H.I.a.e13.Diarrea',
    'H.I.a.e14.Anorexia o nauseas o vómitos',
    'H.I.a.e15.Cefalea',
    'H.I.b.e1.Prueba PCR para SARS-CoV-2 positiva',
    'H.I.b.e2.Prueba de antígenos para SARS-CoV-2 positiva',
    'H.I.c.e1.Tomografía de tórax con opacidades bilaterales múltiples en vidrio esmerilado, con distribución pulmonar periférica y baja sin otra causa conocida',
    'H.II.a.e1.Fiebre igual o mayor a 37,8 °C axilar',
    'H.II.a.e2.Perdida brusca y completa del olfato (anosmia)',
    'H.II.a.e3. Perdida brusca y completa del gusto (ageusia)',
    'H.II.a.e4.Tos o estornudos',
    'H.II.a.e5.Congestión nasal',
    'H.II.a.e6.Disnea o dificultad respiratoria',
    'H.II.a.e7.Taquipnea',
    'H.II.a.e8.Odinofagia',
    'H.II.a.e9.Mialgia',
    'H.II.a.e10.Debilidad general o fatiga',
    'H.II.a.e11.Dolor torácico',
    'H.II.a.e12.Calofríos',
    'H.II.a.e13.Diarrea',
    'H.II.a.e14.Anorexia o nauseas o vómitos',
    'H.II.a.e15.Cefalea',
    'H.II.b.e1.Prueba PCR para SARS-CoV-2 positiva',
    'H.II.b.e2.Prueba de antígenos para SARS-CoV-2 positiva',
    # I - Endometritis
    'I.I.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'I.I.a.e2.Sensibilidad uterina o subinvolución uterina',
    'I.I.a.e3.Loquios de aspecto purulento o cambio en la evolución de su aspecto o aumento de mal olor',
    'I.II.a.e1.La paciente tiene un cultivo de fluido o tejido endometrial positivo obtenidos intraoperatoriamente, por punción uterina o por aspirado uterino con técnica aséptica hasta 10 días posterior al parto',
    # J - Meningitis/Ventriculitis
    'J.I.a.e1.Detección de microorganismos (cultivo, test molecular) en líquido cefalorraquídeo (LCR) recolectado con técnica aséptica para fines diagnósticos o terapéuticos',
    'J.II.a.e1.Fiebre igual o mayor a 38 °C axilar',
    'J.II.a.e2.Dolor de cabeza',
    'J.II.a.e3.Signos meníngeos',
    'J.II.a.e4.Signos de nervios craneales',
    'J.II.a.e5.Modificación cualitativa o cuantitativa de conciencia.',
    'J.II.a.e6.Apnea (en menores de un año)',
    'J.II.a.e7.Bradicardia (en menores de un año)',
    'J.II.b.e1.LCR con aumento de glóbulos blancos o en los niveles de proteínas o con descenso de nivel de glucosa según rangos reportados por laboratorio local',
    'J.II.b.e2.Microorganismo observados en tinción de Gram del LCR',
    'J.II.b.e3.Identificación en uno o más set de hemocultivos periféricos de un microorganismo no relacionado con otra infección activa en otra localización por el mismo agente',
    # K - Endoftalmitis
    'K.I.a.e1.Paciente presenta identificación de un microorganismo en muestra tomada con técnica aséptica de cámara anterior, posterior o humor vítreo',
    'K.II.a.e1.Dolor ocular',
    'K.II.a.e2.Visión borrosa',
    'K.II.a.e3.Hipopion',
    'K.II.b.e1.Como consecuencia de los signos y síntomas, el médico inicia terapia antibiótica de 2 o más días de duración',
]

# Conjunto para búsqueda eficiente
CRITERIOS_SET = {c.strip().lower() for c in CRITERIOS_VALIDOS}

# Mapeo de letra inicial a categoría de infección
CATEGORIA_CRITERIO = {
    'a': 'bacteriemia_cvc',
    'b': 'itu_sonda',
    'c': 'isq',
    'd': 'diarrea_infecciosa',
    'e': 'c_difficile',
    'f': 'neumonia',
    'g': 'ira_viral',
    'h': 'covid19',
    'i': 'endometritis',
    'j': 'meningitis',
    'k': 'endoftalmitis',
}

COLS_CRITERIO = ['Criterio 1', 'Criterio 2', 'Criterio 3', 'Criterio 4', 'Criterio 5']
VALORES_NO_CRITERIO = {'sin hallazgos iaas', 'no aplica', 'nan', 'sin_dato', ''}

def es_criterio_valido(valor):
    v = str(valor).strip().lower()
    return v not in VALORES_NO_CRITERIO and v in CRITERIOS_SET

def calcular_features_criterios(fila):
    conteo = 0
    categorias = {}
    for col in COLS_CRITERIO:
        val = fila[col]
        if es_criterio_valido(val):
            conteo += 1
            pref = str(val).strip()[0].lower() if str(val).strip() else ''
            if pref in CATEGORIA_CRITERIO:
                cat = CATEGORIA_CRITERIO[pref]
                categorias[cat] = categorias.get(cat, 0) + 1
    n_categorias = len(categorias)
    cat_principal = max(categorias, key=categorias.get) if categorias else 'ninguna'
    return conteo, n_categorias, cat_principal

results = df.apply(calcular_features_criterios, axis=1)
df['n_criterios']         = [r[0] for r in results]
df['n_cat_criterios']     = [r[1] for r in results]
df['categoria_principal'] = [r[2] for r in results]

df['tiene_araisp'] = df['ARAISP'].apply(
    lambda x: 0 if pd.isna(x) or str(x).strip().lower() in ['nan', '', 'no', 'no aplica'] else 1
)

def categorizar_dispositivo(tipo):
    tipo = str(tipo).lower()
    if 'catéter venoso central' in tipo or 'cvc' in tipo:
        return 'cvc'
    elif 'sonda vesical' in tipo:
        return 'sonda_vesical'
    elif 'ventilación' in tipo:
        return 'ventilacion_mecanica'
    elif 'epicutáneo' in tipo:
        return 'cateter_epicutaneo'
    elif 'hemodiálisis' in tipo or 'hemodialisis' in tipo:
        return 'cateter_hemodialisis'
    elif 'umbilical' in tipo:
        return 'cateter_umbilical'
    elif 'derivativa' in tipo or 'dve' in tipo or 'dvp' in tipo:
        return 'derivativa_ventricular'
    else:
        return 'otro'

df['tipo_dispositivo_cat'] = df['Tipo de invasivo'].apply(categorizar_dispositivo)

print("✅ Variables derivadas creadas (criterios validados con lista oficial IAAS)")
print(f"  n_criterios (válidos):     {df['n_criterios'].value_counts().sort_index().to_dict()}")
print(f"  n_cat_criterios distintas: {df['n_cat_criterios'].value_counts().sort_index().to_dict()}")
print(f"  categoria_principal:       {df['categoria_principal'].value_counts().to_dict()}")
print(f"  tipo_dispositivo_cat:      {df['tipo_dispositivo_cat'].value_counts().to_dict()}")


In [ ]:
# --- 5.3 Codificación de variables categóricas ---
FEATURES_CATEGORICAS = ['Servicio', 'Procedencia', 'Destino', 'tipo_dispositivo_cat', 'categoria_principal']
FEATURES_NUMERICAS   = ['n_criterios', 'n_cat_criterios', 'tiene_araisp', 'anio']

encoders = {}
df_modelo = df.copy()

for col in FEATURES_CATEGORICAS:
    le = LabelEncoder()
    df_modelo[col + '_enc'] = le.fit_transform(df_modelo[col].astype(str))
    encoders[col] = le
    print(f"  {col}: {len(le.classes_)} categorías → codificadas 0 a {len(le.classes_)-1}")

FEATURES = [col + '_enc' for col in FEATURES_CATEGORICAS] + FEATURES_NUMERICAS
TARGET   = 'IAAS_binaria'

X = df_modelo[FEATURES]
y = df_modelo[TARGET]

print(f"\n✅ Dataset final: {len(X)} registros, {len(FEATURES)} features")
print(f"  Features: {FEATURES}")


---
## ✂️ PASO 6: División 80% / 20%

Usamos `stratify=y` para mantener la proporción de clases en ambas particiones. Guardamos los índices del set de prueba para poder recuperar las columnas originales al exportar el Excel.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("📊 División completada:")
print(f"  Entrenamiento: {len(X_train)} registros ({len(X_train)/len(X)*100:.1f}%)")
print(f"    - IAAS=0 (No): {(y_train==0).sum()} | IAAS=1 (Sí): {(y_train==1).sum()}")
print(f"  Prueba:         {len(X_test)} registros ({len(X_test)/len(X)*100:.1f}%)")
print(f"    - IAAS=0 (No): {(y_test==0).sum()} | IAAS=1 (Sí): {(y_test==1).sum()}")

---
## ⚖️ PASO 7: Balanceo con SMOTE

⚠️ SMOTE se aplica **solo al set de entrenamiento** para no contaminar la evaluación.

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Antes de SMOTE   → IAAS=0:", (y_train==0).sum(), "| IAAS=1:", (y_train==1).sum())
print("Después de SMOTE → IAAS=0:", (y_train_bal==0).sum(), "| IAAS=1:", (y_train_bal==1).sum())
print(f"Total entrenamiento balanceado: {len(X_train_bal)} registros")

---
## 🌲 PASO 8: Entrenamiento del modelo

In [ ]:
print("🌲 Entrenando Random Forest Balanceado...")
modelo = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
modelo.fit(X_train_bal, y_train_bal)
print(f"✅ Modelo entrenado — {modelo.n_estimators} árboles, {modelo.n_features_in_} features")

---
## 📊 PASO 9: Evaluación — Métricas

### ¿Qué mide cada métrica?
- **Accuracy**: % de predicciones correctas en total
- **Precisión**: De los que predijo como IAAS=Sí, ¿cuántos realmente lo son?
- **Sensibilidad (Recall)**: De todos los IAAS reales, ¿cuántos detectó el modelo?
- **F1-Score**: Media armónica entre Precisión y Recall
- **AUC-ROC**: Capacidad discriminativa (1.0 = perfecto, 0.5 = azar)

In [ ]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
auc_roc   = roc_auc_score(y_test, y_prob)
cm        = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Umbral óptimo (Índice de Youden)
fpr_vals, tpr_vals, thresholds = roc_curve(y_test, y_prob)
idx_optimo    = np.argmax(tpr_vals - fpr_vals)
umbral_optimo = float(thresholds[idx_optimo])

print("=" * 54)
print("           MÉTRICAS DE EVALUACIÓN")
print("=" * 54)
print(f"  Accuracy   (exactitud global):      {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  Precision  (precisión IAAS=Sí):     {precision:.4f}  ({precision*100:.1f}%)")
print(f"  Recall     (sensibilidad IAAS=Sí):  {recall:.4f}  ({recall*100:.1f}%)")
print(f"  F1-Score   (balance Prec/Recall):   {f1:.4f}")
print(f"  AUC-ROC    (discriminación):        {auc_roc:.4f}")
print(f"  Umbral óptimo (Youden):             {umbral_optimo:.3f}")
print("=" * 54)
print()
print("📋 Reporte completo por clase:")
print(classification_report(y_test, y_pred, target_names=['No IAAS', 'IAAS']))

---
## 🔲 PASO 10: Matriz de Confusión

|  | Predicho: No IAAS | Predicho: IAAS |
|---|---|---|
| **Real: No IAAS** | Verdadero Negativo (VN) ✅ | Falso Positivo (FP) ⚠️ |
| **Real: IAAS** | Falso Negativo (FN) ❌ | Verdadero Positivo (VP) ✅ |

En vigilancia clínica, los **Falsos Negativos** son críticos: casos de IAAS que el modelo no detectó.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel izquierdo: conteos absolutos ---
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No IAAS', 'IAAS'],
    yticklabels=['No IAAS', 'IAAS'],
    ax=axes[0], linewidths=0.5, annot_kws={'size': 14}
)
axes[0].set_title('Matriz de Confusión\n(conteos absolutos)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Clase Predicha', fontsize=11)
axes[0].set_ylabel('Clase Real', fontsize=11)

labels_mc = [['VN\n(Verdaderos\nNegativos)', 'FP\n(Falsos\nPositivos)'],
             ['FN\n(Falsos\nNegativos)', 'VP\n(Verdaderos\nPositivos)']]
for i in range(2):
    for j in range(2):
        axes[0].text(j + 0.5, i + 0.78, labels_mc[i][j],
                     ha='center', va='center', fontsize=7.5, color='gray')

# --- Panel derecho: porcentajes por fila ---
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='Greens',
    xticklabels=['No IAAS', 'IAAS'],
    yticklabels=['No IAAS', 'IAAS'],
    ax=axes[1], linewidths=0.5, annot_kws={'size': 13}
)
axes[1].set_title('Matriz de Confusión\n(porcentajes por fila)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Clase Predicha', fontsize=11)
axes[1].set_ylabel('Clase Real', fontsize=11)
for text in axes[1].texts:
    text.set_text(text.get_text() + '%')

plt.suptitle('Random Forest Balanceado — Matriz de Confusión (Set de Prueba)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📋 Interpretación:")
print(f"  ✅ Verdaderos Negativos (VN): {tn}  → sin IAAS, correctamente clasificados")
print(f"  ⚠️  Falsos Positivos    (FP): {fp}  → alertas falsas (sin IAAS, predicho como IAAS)")
print(f"  ❌ Falsos Negativos    (FN): {fn}  → IAAS no detectados (riesgo clínico)")
print(f"  ✅ Verdaderos Positivos (VP): {tp}  → casos de IAAS correctamente detectados")
print()
print(f"  Sensibilidad (detección IAAS):     {recall*100:.1f}%")
print(f"  Especificidad (detección No IAAS): {tn/(tn+fp)*100:.1f}%")

---
## 📈 PASO 11: Gráficos de Confianza — Curva ROC y Distribución de Probabilidades

- **Curva ROC**: muestra el trade-off entre sensibilidad y especificidad para distintos umbrales. AUC=1.0 es perfecto.
- **Distribución de probabilidades**: qué tan bien separa el modelo las dos clases. Mayor separación = mayor confianza.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Curva ROC ---
axes[0].plot(fpr_vals, tpr_vals, color='steelblue', lw=2.5,
             label=f'Random Forest (AUC = {auc_roc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--',
             label='Clasificador aleatorio (AUC = 0.5)')
axes[0].fill_between(fpr_vals, tpr_vals, alpha=0.10, color='steelblue')

# Punto óptimo (máximo Youden)
axes[0].scatter(fpr_vals[idx_optimo], tpr_vals[idx_optimo],
                color='red', s=100, zorder=5,
                label=f'Umbral óptimo = {umbral_optimo:.2f}')

axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.02])
axes[0].set_xlabel('Tasa de Falsos Positivos (1 − Especificidad)', fontsize=11)
axes[0].set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=11)
axes[0].set_title('Curva ROC\nRandom Forest Balanceado', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- Distribución de probabilidades predichas ---
prob_no_iaas = y_prob[y_test == 0]
prob_si_iaas = y_prob[y_test == 1]

axes[1].hist(prob_no_iaas, bins=30, alpha=0.6, color='steelblue',
             label='No IAAS (real)', density=True, edgecolor='white')
axes[1].hist(prob_si_iaas, bins=30, alpha=0.6, color='tomato',
             label='IAAS (real)', density=True, edgecolor='white')
axes[1].axvline(x=0.5, color='black', linestyle='--', lw=2, label='Umbral = 0.5')
axes[1].axvline(x=umbral_optimo, color='red', linestyle=':', lw=2,
                label=f'Umbral óptimo = {umbral_optimo:.2f}')

axes[1].set_xlabel('Probabilidad Predicha de IAAS', fontsize=11)
axes[1].set_ylabel('Densidad', fontsize=11)
axes[1].set_title('Distribución de Probabilidades Predichas\n(confianza del modelo por clase)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Gráficos de Confianza — Random Forest Balanceado',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('grafico_confianza.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"🎯 Umbral óptimo (Índice de Youden): {umbral_optimo:.3f}")
print("   → Probabilidad ≥ umbral → IAAS=Sí")

---
## 🏆 PASO 12: Importancia de Variables

Random Forest calcula la importancia de cada variable según cuánto reduce la impureza de Gini al dividir los nodos. Mayor valor = más relevante para predecir IAAS.

In [ ]:
nombres_legibles = {
    'Servicio_enc':              'Servicio (Unidad clínica)',
    'Procedencia_enc':           'Procedencia del paciente',
    'Destino_enc':               'Destino del paciente',
    'tipo_dispositivo_cat_enc':  'Tipo de dispositivo invasivo',
    'categoria_principal_enc':   'Categoría IAAS principal',
    'n_criterios':               'N° criterios válidos (lista oficial)',
    'n_cat_criterios':           'N° categorías IAAS distintas',
    'tiene_araisp':              'Tiene ARAISP',
    'anio':                      'Año'
}

importancias_df = pd.DataFrame({
    'Variable':    FEATURES,
    'Importancia': modelo.feature_importances_
}).sort_values('Importancia', ascending=True)

importancias_df['Variable_legible'] = importancias_df['Variable'].map(
    lambda x: nombres_legibles.get(x, x)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    importancias_df['Variable_legible'],
    importancias_df['Importancia'],
    color=plt.cm.viridis(importancias_df['Importancia'] / importancias_df['Importancia'].max()),
    edgecolor='white', height=0.6
)
for bar, val in zip(bars, importancias_df['Importancia']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

ax.set_xlabel('Importancia (reducción de impureza de Gini)', fontsize=11)
ax.set_title('Importancia de Variables — Random Forest Balanceado', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, importancias_df['Importancia'].max() * 1.2)
plt.tight_layout()
plt.savefig('importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Importancia de variables (mayor = más relevante):")
print(importancias_df[['Variable_legible', 'Importancia']]
      .sort_values('Importancia', ascending=False)
      .to_string(index=False))


---
## 📋 PASO 13: Resumen Ejecutivo

In [ ]:
mejor_var = importancias_df.sort_values('Importancia', ascending=False).iloc[0]

print("=" * 62)
print("       RESUMEN EJECUTIVO — MODELO RANDOM FOREST IAAS")
print("=" * 62)
print()
print("📂 DATOS")
print(f"  Registros totales:            {len(df):,}")
print(f"  Entrenamiento (con SMOTE):    {len(X_train_bal):,}")
print(f"  Prueba (sin balanceo):        {len(X_test):,}")
print(f"  Período:                      2024 – 2025")
print()
print("🌲 MODELO")
print(f"  Algoritmo:    Random Forest Balanceado")
print(f"  Balanceo:     SMOTE + class_weight='balanced'")
print(f"  Árboles:      300  |  max_depth=10")
print()
print("📊 MÉTRICAS (set de prueba no visto)")
print(f"  Accuracy:   {accuracy:.4f}  ({accuracy*100:.1f}%)")
print(f"  Precision:  {precision:.4f}  ({precision*100:.1f}%)")
print(f"  Recall:     {recall:.4f}  ({recall*100:.1f}%)")
print(f"  F1-Score:   {f1:.4f}")
print(f"  AUC-ROC:    {auc_roc:.4f}")
print()
print("🔲 MATRIZ DE CONFUSIÓN")
print(f"  VN={tn}  FP={fp}  FN={fn}  VP={tp}")
print(f"  Sensibilidad:   {tp/(tp+fn)*100:.1f}%  |  Especificidad: {tn/(tn+fp)*100:.1f}%")
print()
print(f"🎯 Variable más importante: {mejor_var['Variable_legible']} ({mejor_var['Importancia']:.3f})")
print()
print("💾 ARCHIVOS A GENERAR")
print("  - modelo_iaas.pkl")
print("  - matriz_confusion.png")
print("  - grafico_confianza.png")
print("  - importancia_variables.png")
print("  - set_prueba_con_predicciones.xlsx")
print("=" * 62)

---
## 📤 PASO 14: Exportar el 20% de prueba a Excel

Se exporta el set de prueba en el **mismo formato exacto** que el Excel de entrada: las mismas 20 columnas en el mismo orden (`Rut`, `Servicio`, `Apellido paterno`, … `IAAS`), copiando las filas directamente desde los datos originales usando el índice guardado. Al final se agregan tres columnas del modelo: probabilidad predicha, clasificación y acierto.

In [ ]:
# Extraer las filas originales del set de prueba usando el índice guardado
df_export = df_raw_completo.iloc[X_test.index].copy().reset_index(drop=True)

# pandas renombra automáticamente el segundo 'Rut' a 'Rut.1' para evitar duplicados.
# Lo dejamos así internamente y controlamos el encabezado visible al escribir el Excel.

# Calcular columna 'Días' = Fecha de retiro - Fecha de instalación
def calcular_dias(row):
    try:
        ret = pd.to_datetime(row['Fecha de retiro'])
        ins = pd.to_datetime(row['Fecha de instalación'])
        if pd.notna(ret) and pd.notna(ins):
            return (ret - ins).days
    except Exception:
        pass
    return None

df_export['Días'] = df_export.apply(calcular_dias, axis=1)

# Predicciones del modelo
y_prob_export = modelo.predict_proba(X_test)[:, 1]
y_pred_optimo = (y_prob_export >= umbral_optimo).astype(int)

df_export['Prob_IAAS (%)']                            = (y_prob_export * 100).round(1)
df_export[f'Prediccion (umbral {umbral_optimo:.2f})'] = ['SÍ IAAS' if p == 1 else 'NO IAAS' for p in y_pred_optimo]
df_export['Acierto']                                  = ['✓ Correcto' if p == r else '✗ Error'
                                                         for p, r in zip(y_pred_optimo, y_test.values)]

# ================================================================
# ORDEN EXACTO: Rut | Rut | Servicio | Apellido paterno | Apellido materno |
# Nombres | Fecha de ingreso | Fecha de egreso | Procedencia | Destino |
# Tipo de invasivo | Fecha de instalación | Fecha de retiro | Días |
# ARAISP | Criterio 1..5 | IAAS  + columnas del modelo
# ================================================================
COLS_INTERNAS = [
    'Rut', 'Rut.1',
    'Servicio', 'Apellido paterno', 'Apellido materno', 'Nombres',
    'Fecha de ingreso', 'Fecha de egreso',
    'Procedencia', 'Destino', 'Tipo de invasivo',
    'Fecha de instalación', 'Fecha de retiro', 'Días',
    'ARAISP',
    'Criterio 1', 'Criterio 2', 'Criterio 3', 'Criterio 4', 'Criterio 5',
    'IAAS',
    'Prob_IAAS (%)', f'Prediccion (umbral {umbral_optimo:.2f})', 'Acierto'
]
# Conservar solo las que existen
COLS_INTERNAS = [c for c in COLS_INTERNAS if c in df_export.columns]
df_export = df_export[COLS_INTERNAS]

# Encabezados visibles: 'Rut.1' → 'Rut'
HEADERS = ['Rut' if c == 'Rut.1' else c for c in COLS_INTERNAS]

# ---- Crear Excel con formato ----
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = Workbook()
ws = wb.active
ws.title = 'Set de Prueba'

# Fila de encabezado con nombres visibles correctos
for c_idx, nombre in enumerate(HEADERS, start=1):
    cell = ws.cell(row=1, column=c_idx, value=nombre)
    cell.font      = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    cell.fill      = PatternFill('solid', start_color='1F4E79')
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# Filas de datos
for r_idx, (_, fila) in enumerate(df_export.iterrows(), start=2):
    for c_idx, value in enumerate(fila, start=1):
        # Convertir NaN/NaT a None para que openpyxl lo escriba como celda vacía
        try:
            if pd.isna(value):
                value = None
        except Exception:
            pass
        cell = ws.cell(row=r_idx, column=c_idx, value=value)
        cell.font      = Font(name='Arial', size=9)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if isinstance(value, str) and '✓' in value:
            cell.fill = PatternFill('solid', start_color='C6EFCE')
            cell.font = Font(name='Arial', size=9, color='276221')
        elif isinstance(value, str) and '✗' in value:
            cell.fill = PatternFill('solid', start_color='FFC7CE')
            cell.font = Font(name='Arial', size=9, color='9C0006')
        elif isinstance(value, str) and value == 'SÍ IAAS':
            cell.fill = PatternFill('solid', start_color='FFE699')
            cell.font = Font(name='Arial', size=9, bold=True, color='7F6000')

ws.row_dimensions[1].height = 38
ws.freeze_panes = 'A2'

# Anchos de columna
anchos = [14, 16, 16, 18, 18, 16, 16, 16, 18, 18, 24, 18, 16, 8, 12,
          22, 22, 22, 22, 22, 10, 14, 26, 14]
for i, w in enumerate(anchos, start=1):
    if i <= len(HEADERS):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = w

ARCHIVO_PRUEBA = 'set_prueba_con_predicciones.xlsx'
wb.save(ARCHIVO_PRUEBA)

import os
print(f"✅ Set de prueba exportado: {ARCHIVO_PRUEBA} ({os.path.getsize(ARCHIVO_PRUEBA)/1024:.1f} KB)")
print(f"   Registros: {len(df_export)}")
print(f"   Columnas ({len(HEADERS)}):")
for h in HEADERS:
    print(f"     · {h}")

---
## 💾 PASO 15: Exportar el modelo entrenado (.pkl)

Genera `modelo_iaas.pkl` con todo lo necesario para el notebook frontend.

In [ ]:
categorias_validas = {}
for col in FEATURES_CATEGORICAS:
    categorias_validas[col] = sorted(encoders[col].classes_.tolist())

importancias_dict = dict(zip(FEATURES, modelo.feature_importances_.tolist()))

paquete_modelo = {
    'modelo':               modelo,
    'encoders':             encoders,
    'features':             FEATURES,
    'features_categoricas': FEATURES_CATEGORICAS,
    'features_numericas':   FEATURES_NUMERICAS,
    'categorias_validas':   categorias_validas,
    'umbral_optimo':        umbral_optimo,
    'metricas': {
        'accuracy':  round(accuracy, 4),
        'precision': round(precision, 4),
        'recall':    round(recall, 4),
        'f1':        round(f1, 4),
        'auc_roc':   round(auc_roc, 4),
        'n_train':   len(X_train_bal),
        'n_test':    len(X_test),
        'n_total':   len(df),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
    },
    'importancias':    importancias_dict,
    'anio_referencia': 2025,
}

ARCHIVO_MODELO = 'modelo_iaas.pkl'
with open(ARCHIVO_MODELO, 'wb') as f:
    pickle.dump(paquete_modelo, f)

import os
print(f"✅ Modelo exportado: {ARCHIVO_MODELO} ({os.path.getsize(ARCHIVO_MODELO)/1024:.1f} KB)")
print()
print("📦 Contenido:")
print(f"  - modelo:            RandomForestClassifier ({modelo.n_estimators} árboles)")
print(f"  - encoders:          {list(encoders.keys())}")
print(f"  - umbral_optimo:     {umbral_optimo:.3f}")
print(f"  - métricas:          accuracy={accuracy:.3f}, AUC-ROC={auc_roc:.3f}")

---
## 📥 PASO 16: Descargar todos los archivos

In [ ]:
from google.colab import files

archivos = [
    'modelo_iaas.pkl',
    'matriz_confusion.png',
    'grafico_confianza.png',
    'importancia_variables.png',
    'set_prueba_con_predicciones.xlsx',
]

for archivo in archivos:
    files.download(archivo)
    print(f"⬇️  Descargando {archivo}...")

print("\n✅ Todos los archivos descargados correctamente")